# Chargement des données

In [ ]:
import duckdb
%load_ext sql

In [ ]:
%sql duckdb:///../data/istex-search-metadata.db

## Echantillonnage

### Hosts

In [ ]:
%%sql
SELECT
  *
FROM
  hosts USING SAMPLE 1000 (reservoir);

## Records

In [ ]:
%%sql
SELECT
  *
FROM
  records USING SAMPLE 1000 (reservoir);

## Metadata

In [ ]:
%%sql
SELECT
  *
FROM
  metadata USING SAMPLE 1000 (reservoir);

# Analyse

In [ ]:
%%sql
SELECT
  *
FROM
  hosts USING SAMPLE 1000 (reservoir);

In [ ]:
%%sql
SELECT
  COUNT(*) AS "Documents créés par la chaine d'ingestion Istex"
FROM
  (
    SELECT
      "path"
    FROM
      metadata
    WHERE
      original = FALSE
    UNION ALL
    SELECT
      "path"
    FROM
      fulltext
    WHERE
      original = FALSE
  );

In [ ]:
%%sql

WITH cte_extension_fulltext AS (
    SELECT 'fulltext' AS "type", LOWER(string_split(path, '.')[-1]) AS extension FROM fulltext
),
cte_extension_metadata AS (
    SELECT 'metadata' AS "type", LOWER(string_split(path, '.')[-1]) AS extension FROM metadata
),
cte_extension AS (
    SELECT * FROM cte_extension_fulltext
    UNION ALL
    SELECT * FROM cte_extension_metadata
)
    
SELECT extension, type, COUNT(*) AS "Nombre de fichiers" FROM cte_extension
GROUP BY extension, type;

# Analyse statistique à faire sur les fichiers OCR produits par Istex

* OCR / par décénnies
* OCR par disciplines
* OCR par langue

# Données à récupérer dans l'API Istex

* Si pas de texte transparent
* Si pas d'image

arkIstex
genre -> du document
language -> du document
publicationDate
accessCondition.value (on pourra inférer le accessCondition.contentType au besoin)
corpusName
metadata ->  ark, path, mime, orignal
fulltext -> ark, path, mime, orignal

## Analyse fichier PDF

10 % du fichier PDF arrodi à l'entier supérieur. Selection aléatoire pour éviter de tomber sur des pages d'annexes

### Au niveau de chaque page (ou d'un échantillon de page)

* Liste des images et page sur laquelle a été trouvée l'image + taille de l'image + taille de la page
* Présence de texte transparent -> compter le nombre de caractères (pour faire des ratios) ? 
* Présence de texte pas transparent -> compter le nombre de caractères
* Présence de texte très fragmenté (Tj) 
* Taille de police unique

* Pourquoi 10% ?
* Graine pour le choix aléatoire des pages.
* Passer tout UK-19 PARLIAMENT
* Autoriser la reprise du script

In [ ]:
%config SqlMagic.displaylimit = 100

In [ ]:
%%sql

SELECT * FROM fulltext JOIN records USING(arkIstex)  WHERE PATH like '%.ocr';

In [ ]:
%%sql
SELECT
    decade(try_strptime(publicationDate, '%Y')) * 10 AS decade,
    COUNT(DISTINCT r.arkIstex) AS nb_total,
    COUNT(DISTINCT CASE WHEN f.path LIKE '%.ocr' THEN f.arkIstex END) AS nb_ocr
FROM records r
LEFT JOIN fulltext f USING(arkIstex)
WHERE try_strptime(publicationDate, '%Y') IS NOT NULL
GROUP BY decade
HAVING decade > 1455 AND decade < 2026
ORDER BY decade;

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import matplotlib as mpl
import numpy as np
import os

df = _.DataFrame()

# Create output folder if needed
if not os.path.exists('Figures'):
    os.mkdir('Figures')

# Create figure
fig, ax1 = plt.subplots(figsize=(12, 6))

# Viridis colormap


cmap = mpl.colormaps['viridis']
line_color = cmap(0.2)
bar_color = cmap(0.7)

# Formatter functions
millions = lambda x, pos: f"{x/1_000_000:.1f}M"
mk = lambda x, pos: f"{x/1_000:.1f}K"

# Left axis — total documents (line)
ax1.plot(df['decade'], df['nb_total'],
         color=line_color, marker='o', linewidth=2, zorder=3,
         label='Nombre total de documents')

ax1.set_xlabel('Décennie')

ax1.tick_params(axis='y', labelcolor=line_color)
ax1.yaxis.set_major_formatter(FuncFormatter(millions))
ax1.grid(axis='y', linestyle='--', alpha=0.3)

# Right axis — OCR documents (bars)
ax2 = ax1.twinx()
ax2.bar(df['decade'], df['nb_ocr'],
        width=5, color=bar_color, alpha=0.8, zorder=2,
        label='Documents OCR')


ax2.tick_params(axis='y', labelcolor=bar_color)
ax2.yaxis.set_major_formatter(FuncFormatter(mk))

# Ensure line is above bars
ax1.set_zorder(2)
ax2.set_zorder(1)
ax1.patch.set_alpha(0)

# Combined legend (line first for readability)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

# X-axis formatting
ax1.set_xticks(df['decade'])
ax1.set_xticklabels(df['decade'], rotation=45, ha='right')
ax1.set_xlim(left=1800, right=2030)

# Title
plt.title('Évolution du nombre de documents océrisés dans ISTEX (par décennies de publication)')

# Layout + save
plt.tight_layout()
fig.savefig('./Figures/evol_ocr_istex.png', dpi=600, bbox_inches='tight')

plt.show()

In [ ]:
%%sql

SELECT
    lower(list_last(split(path, '.'))) AS extension,
    COUNT(*) AS count
FROM fulltext
GROUP BY extension, mime
HAVING mime = 'text/plain'
ORDER BY count DESC;

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

import os

df = _.DataFrame()

if not os.path.exists('Figures'):
    os.mkdir('Figures')

# Viridis color palette
cmap = mpl.colormaps['viridis']
colors = [cmap(i) for i in np.linspace(0.2, 0.8, len(df))]

# Figure
plt.figure(figsize=(12, 10))

# Donut chart with labels outside
wedges, texts = plt.pie(
    df['count'],
    labels=None,  # We'll add custom labels outside manually
    startangle=90,
    colors=colors,
    wedgeprops={'edgecolor': 'w', 'linewidth': 1, 'width': 0.4}
)

# Custom labels outside with percentages and counts
total = df['count'].sum()
for i, p in enumerate(wedges):
    ang = (p.theta2 - p.theta1)/2. + p.theta1
    y = np.sin(np.deg2rad(ang))
    x = np.cos(np.deg2rad(ang))
    horizontalalignment = 'left' if x > 0 else 'right'
    connectionstyle = f"angle,angleA=0,angleB={ang}"
    
    plt.annotate(
        f"{df['extension'][i]}: {df['count'][i]:,} ({df['count'][i]/total*100:.1f}%)",
        xy=(x*0.7, y*0.7),
        xytext=(1.2*x, 1.2*y),
        horizontalalignment=horizontalalignment,
        fontsize=12,
        weight='bold',
        arrowprops=dict(arrowstyle='-', color='gray', connectionstyle=connectionstyle)
    )

# Center total
plt.text(0, 0, f"Total\n{total:,}", ha='center', va='center', fontsize=16, weight='bold')

# Title
plt.title('Répartition des fichiers par extension dans ISTEX', fontsize=18, pad=20, weight='bold')

plt.tight_layout()
plt.savefig('./Figures/donut_extensions_improved.png', dpi=600, bbox_inches='tight')
plt.show()